## Extract text from pdf/docx Resume

In [2]:
import os
import pdfplumber
import docx
from tqdm import tqdm

# Folder paths
input_folder = "resumes_raw"
output_folder = "resumes_text"

# Create output folder if not exists
os.makedirs(output_folder, exist_ok=True)

# Function to read PDF
def extract_pdf_text(file_path):
    text = ""
    with pdfplumber.open(file_path) as pdf:
        for page in pdf.pages:
            text += page.extract_text() or ""
    return text

# Function to read DOCX
def extract_docx_text(file_path):
    doc = docx.Document(file_path)
    text = "\n".join([para.text for para in doc.paragraphs])
    return text

# Get all resume files
files = os.listdir(input_folder)

# Process each resume
for file in tqdm(files):

    file_path = os.path.join(input_folder, file)

    if file.endswith(".pdf"):
        text = extract_pdf_text(file_path)

    elif file.endswith(".docx"):
        text = extract_docx_text(file_path)

    else:
        continue

    output_file = os.path.join(
        output_folder,
        file.replace(".pdf", ".txt").replace(".docx", ".txt")
    )

    with open(output_file, "w", encoding="utf-8") as f:
        f.write(text)

print("Extraction Successful")

  2%|█▋                                                                                | 7/349 [00:02<02:14,  2.54it/s]Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
 15%|████████████                                                                     | 52/349 [00:25<03:16,  1.51it/s]Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
 24%|███████████████████▋                                                 

Extraction Successful


### Building dataset/table from extracted text

In [8]:
import os
import pandas as pd

resume_folder = "resumes_text"

data = []

for file in os.listdir(resume_folder):

    path = os.path.join(resume_folder, file)

    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        text = f.read().lower()

    # ---------- Extract Candidate Name ----------
    lines = text.split("\n")
    candidate_name = ""

    for line in lines[:10]:
        line = line.strip()

        if len(line) > 3 and len(line) < 40:
            if not any(word in line for word in ["resume","cv","email","phone","contact"]):
                candidate_name = line
                break

    if candidate_name == "":
        candidate_name = file.replace(".txt","")

    # ---------- Skills ----------
    skills_keywords = [
        "python","sql","excel","power bi","tableau",
        "html","css","javascript","react","node",
        "photoshop","figma","illustrator","canva",
        "marketing","seo", "social media marketing",  "content marketing",
        "communication","recruitment", "management","excel"
    ]

    skills_found = [skill for skill in skills_keywords if skill in text]

    # ---------- Tools ----------
    tools_keywords = [
        "power bi","tableau","figma","canva",
        "photoshop","excel","git"
    ]

    tools_found = [tool for tool in tools_keywords if tool in text]

    # ---------- Education ----------
    education = ""

    if "b.tech" in text or "btech" in text:
        education = "B.Tech"
    elif "bca" in text:
        education = "BCA"
    elif "mca" in text:
        education = "MCA"
    elif "bsc" in text:
        education = "B.Sc"

    # ---------- Projects ----------
    projects = "mentioned" if "project" in text else "not mentioned"

    # ---------- Experience ----------
    experience = "mentioned" if "experience" in text or "internship" in text else "not mentioned"

    # ---------- Certification ----------
    certification = "mentioned" if "certification" in text or "certificate" in text else "not mentioned"

    data.append({
        "Candidate_Name": candidate_name,
        "Education": education,
        "Skills": ", ".join(skills_found),
        "Tools": ", ".join(tools_found),
        "Past_Projects": projects,
        "Experience": experience,
        "Certification": certification
    })

df = pd.DataFrame(data)

# Ensure dataset folder exists
os.makedirs("dataset", exist_ok=True)

df.to_excel("dataset/resume_dataset.xlsx", index=False)

print("✅ Dataset created successfully!")

✅ Dataset created successfully!


In [10]:
import pandas as pd
resume_ds=pd.read_excel("dataset/resume_dataset.xlsx")
resume_ds.head()

,Candidate_Name,Education,Skills,Tools,Past_Projects,Experience,Certification
0,shashank,B.Tech,"python, excel, excel",excel,mentioned,mentioned,not mentioned
1,surishetty harivallabha sai,B.Tech,"python, sql, html, css, javascript, photoshop,...","figma, photoshop, git",mentioned,mentioned,mentioned
2,premkumar narayan tambekar,B.Tech,"python, sql, excel, power bi, react, figma, ca...","power bi, figma, canva, excel, git",mentioned,mentioned,mentioned
3,candidate name: dasari harshini,B.Tech,python,NaN,mentioned,mentioned,not mentioned
4,suraj kumar (cid:131) +91 9122601214,NaN,"python, sql, excel, react, node, figma, canva,...","figma, canva, excel, git",mentioned,mentioned,not mentioned


### Creating role based matching datset(only skills considered)

In [19]:
import pandas as pd

# Load datasets
candidate_df = pd.read_excel("dataset/resume_dataset.xlsx")
role_df = pd.read_excel("dataset/role_skill_matrix.xlsx")

results = []

for _, candidate in candidate_df.iterrows():

    candidate_name = candidate["Candidate_Name"]
    candidate_skills = str(candidate["Skills"]).lower().split(",")

    candidate_skills = [skill.strip() for skill in candidate_skills]

    for _, role in role_df.iterrows():

        role_name = role["Role"]
        role_skills = role["Required_Skills"].lower().split(",")

        role_skills = [skill.strip() for skill in role_skills]

        matched = len(set(candidate_skills) & set(role_skills))
        total = len(role_skills)

        match_percentage = round((matched / total) * 100, 2)

        # Fit category thresholds
        if match_percentage >= 70:
            fit_category = "Strong Fit"
        elif match_percentage >= 40:
            fit_category = "Moderate Fit"
        else:
            fit_category = "Low Fit"

        results.append({
            "Candidate_Name": candidate_name,
            "Role": role_name,
            "Match_Percentage": match_percentage,
            "Fit_Category": fit_category
        })

result_df = pd.DataFrame(results)

# Save results
result_df.to_excel("dataset/candidate_role_match.xlsx", index=False)

print("Rule-based matching with fit categories completed!")

Rule-based matching with fit categories completed!


### Advanced:Creating role based matching datset(skills + experience + projects considered)

In [20]:
import pandas as pd

candidate_df = pd.read_excel("dataset/resume_dataset.xlsx")
role_df = pd.read_excel("dataset/role_skill_matrix.xlsx")

results = []

for _, candidate in candidate_df.iterrows():
    candidate_name = candidate["Candidate_Name"]
    candidate_skills = str(candidate["Skills"]).lower().split(",")
    candidate_skills = [skill.strip() for skill in candidate_skills]

    exp_flag = 1 if str(candidate.get("Experience", "not_mentioned")).strip().lower() == "mentioned" else 0
    proj_flag = 1 if str(candidate.get("Past_Projects", "not_mentioned")).strip().lower() == "mentioned" else 0

    for _, role in role_df.iterrows():
        role_name = role["Role"]
        role_skills = role["Required_Skills"].lower().split(",")
        role_skills = [skill.strip() for skill in role_skills]

        matched = len(set(candidate_skills) & set(role_skills))
        total = len(role_skills)

        skill_match_percentage = (matched / total * 100) if total > 0 else 0

        # Weighted scoring
        final_score = (skill_match_percentage * 0.7) + (exp_flag * 20) + (proj_flag * 10)

        # Fit category thresholds
        if final_score >= 70:
            fit_category = "Strong Fit"
        elif final_score >= 40:
            fit_category = "Moderate Fit"
        else:
            fit_category = "Low Fit"

        results.append({
            "Candidate_Name": candidate_name,
            "Role": role_name,
            "Match_Percentage": round(final_score, 2),
            "Fit_Category": fit_category,
            "Experience_Flag": "mentioned" if exp_flag else "not_mentioned",
            "Project_Flag": "mentioned" if proj_flag else "not_mentioned"
        })

result_df = pd.DataFrame(results)
result_df.to_excel("dataset/candidate_role_match_with_experience_project.xlsx", index=False)

print("Weighted rule-based matching completed!")

Weighted rule-based matching completed!
